# Day 3 — Instruction Tuning on Your Own Data

---

Yesterday's 6-example dataset was a toy. Today we build a **real** fine-tune on a public dataset (~1000 examples), then talk about how to shape your own.

Same GPU environment as yesterday (Colab T4).


## 1. Pick a public dataset

Hugging Face hosts thousands of instruction-tuning datasets. Good starter picks:

| Dataset | Domain | Size |
|---|---|---|
| `databricks/databricks-dolly-15k` | General instructions | 15k |
| `HuggingFaceH4/no_robots` | Human-written, high quality | 10k |
| `nvidia/OpenMathInstruct-2` | Math word problems | 2M (use a slice!) |
| `mlabonne/orca-math-word-problems-80k` | Math | 80k |

**For your first serious fine-tune: pick something ~1000-5000 examples.** Bigger takes hours on a T4.


## 2. Load & inspect


In [ ]:
from datasets import load_dataset

ds = load_dataset("databricks/databricks-dolly-15k", split="train[:1000]")
print(ds[0])


Most datasets don't come in your model's chat template. **You always convert.**


In [ ]:
def dolly_to_chat(row):
    system = "You are a helpful assistant."
    user = row["instruction"] + (f"\n\n{row['context']}" if row["context"] else "")
    return {"messages": [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": row["response"]},
    ]}

ds = ds.map(dolly_to_chat, remove_columns=ds.column_names)
print(ds[0]["messages"])


## 3. Train / eval split


In [ ]:
ds = ds.train_test_split(test_size=0.1, seed=42)
print(f"train: {len(ds['train'])}  eval: {len(ds['test'])}")


## 4. Same training loop as Day 2 (with eval added)


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Llama-3.2-3B-Instruct", max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

def fmt(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}

train_ds = ds["train"].map(fmt)
eval_ds  = ds["test"].map(fmt)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir="dolly-out",
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        eval_strategy="steps", eval_steps=50,
        logging_steps=10, save_strategy="no",
        report_to="none", fp16=True,
    ),
)
trainer.train()


**Notice `eval_steps=50`.** Every 50 steps the trainer runs the eval set and reports `eval_loss`. Watch this — if `train_loss` keeps dropping but `eval_loss` starts climbing, you're **overfitting**. Stop, or try more data / more regularization.


## 5. Data quality tips for your own dataset

- **Deduplicate.** Same input twice trains the model to over-predict that output.
- **Filter empties.** `response.strip() == ""` → drop.
- **Length caps.** Drop examples longer than your `max_seq_length`. Training on truncated outputs teaches the model to stop mid-sentence.
- **Balance labels/classes.** If 80% of your training tickets are `billing`, the fine-tune will love saying `billing`.
- **A few dozen manually-reviewed examples beat thousands of scraped ones.**


## 6. When something isn't working

| Symptom | Likely cause | Try |
|---|---|---|
| Train loss doesn't drop | LR too low, bad data | LR 5e-4; inspect a few examples |
| Train loss → 0 fast, eval bad | Overfit | Fewer epochs; more/better data |
| Model outputs garbage | Wrong chat template | Use `apply_chat_template` |
| Model just repeats system prompt | Not enough epochs, or system+user together too long | Check example lengths |
| OOM (out of memory) | Batch too big or seq too long | Lower `batch_size` or `max_seq_length` |


## Recap

- **Convert every dataset** into your model's chat format.
- **Always eval.** `eval_strategy="steps"` catches overfitting early.
- **Data quality dominates** — dedupe, filter, balance, spot-check.
- Common failure modes are cheap to diagnose if you watch loss curves.
- **Next class:** W&B for tracking experiments + honest evaluation.
